## XBRL US API - Python example  
This sample Python code queries the XBRL US Public Filings Database; it is based on a notebook created by [Ties de Kok](https://www.tiesdekok.com).
### Authenticate for access token 
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [74]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
api = input('Enter subdomain ("api" or left blank to query public results, otherwise enter a value) ') or 'api'
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)



Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 

For now, skip ahead to the section 'Make a Query'.


access token: e3507821-d500-409d-894b-aa4b7027ea69 refresh token: 48fc58da-66e1-43a0-8bef-1d96d487b9c4


### Make a query 
After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run_** menu option from the cell **immediately below this text** to run the entire query for results.

The sample results are from 10+ years of data for companies in an SIC code, and may take several minutes to recreate. **To test for results quickly, modify the _params_** to comment out report.sic-code and uncomment entity.cik and period.fiscal-year so the search runs for several companies across a few years.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [75]:
# Define the parameters of the query - this query returns all of the most-recent
# reported fiscal year values for all years as defined below (XBRL_Elements)
# in companies reporting with SIC code 2080

endpoint = 'assertion'
XBRL_Elements = [
    'DQC.US.0099.9533',
    ]
report_year = [
    '2023',       
    '2024'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    'report.entry-url',
    'assertion.code.sort(ASC)',
    'report.base-taxonomy',
    'report.document-type',
    'assertion.run-date',
    'report.accepted-timestamp.sort(DESC)',
    'report.accession',
    'entity.code',
    'entity.name',
    'assertion.type',
    #'assertion.detail',
    'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 20 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.code': ','.join(XBRL_Elements), 
    'report.filing-year': ','.join(report_year),
    'fields': ','.join(fields)
    }

print('\n\nclick the run button below to execute this query')



click the run button below to execute this query


In [76]:
# @title
# ### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Dec 16 14:52:53 2024 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 2654 records.

At Mon Dec 16 14:52:54 2024, the query finished with   2654   rows returned in 0:00:00.181345 for 
https://api.xbrl.us/api/v1/assertion/search?unique&assertion.code=DQC.US.0099.9533&report.filing-year=2023,2024&fields=report.entry-url,assertion.code.sort(ASC),report.base-taxonomy,report.document-type,assertion.run-date,report.accepted-timestamp.sort(DESC),report.accession,entity.code,entity.name,assertion.type,assertion.limit()


,report.entry-url,assertion.code,report.base-taxonomy,report.document-type,assertion.run-date,report.accepted-timestamp,report.accession,entity.code,entity.name,assertion.type
0,http://www.sec.gov/Archives/edgar/data/1840317/000121390024108485/ea0224057-s1_veeainc.htm,DQC.US.0099.9533,US GAAP 2024,S-1,2024-12-13,2024-12-13 06:04:00,0001213900-24-108485,0001840317,VEEA INC.,0099
1,http://www.sec.gov/Archives/edgar/data/1980295/000198029524000021/ima10q_31oct2024.htm,DQC.US.0099.9533,US GAAP 2024,10-Q,2024-12-12,2024-12-12 16:38:00,0001980295-24-000021,0001980295,IMA TECH,0099
2,http://www.sec.gov/Archives/edgar/data/1346610/000121390024108286/ea0224177-posam1_soslimited.htm,DQC.US.0099.9533,US GAAP 2024,POS AM,2024-12-12,2024-12-12 15:01:00,0001213900-24-108286,0001346610,SOS Limited,0099
3,http://www.sec.gov/Archives/edgar/data/1504239/000119983524000544/pcnt-10q.htm,DQC.US.0099.9533,US GAAP 2023,10-Q,2024-12-12,2024-12-12 12:42:00,0001199835-24-000544,0001504239,"POINT OF CARE NANO-TECHNOLOGY, INC.",0099
4,http://www.sec.gov/Archives/edgar/data/1474167/000147793224007979/cosm_s1a.htm,DQC.US.0099.9533,US GAAP 2024,S-1/A,2024-12-11,2024-12-10 16:59:00,0001477932-24-007979,0001474167,COSMOS HEALTH INC.,0099
5,http://www.sec.gov/Archives/edgar/data/74046/000007404624000094/odc-20241031.htm,DQC.US.0099.9533,US GAAP 2023,10-Q,2024-12-09,2024-12-09 16:08:00,0000074046-24-000094,0000074046,OIL-DRI CORPORATION OF AMERICA,0099
6,http://www.sec.gov/Archives/edgar/data/1634117/000163411724000130/bned-20241026.htm,DQC.US.0099.9533,US GAAP 2024,10-Q,2024-12-09,2024-12-09 09:06:00,0001634117-24-000130,0001634117,"BARNES & NOBLE EDUCATION, INC.",0099
7,http://www.sec.gov/Archives/edgar/data/1840317/000121390024106542/ea0222796-s1_veeainc.htm,DQC.US.0099.9533,US GAAP 2024,S-1,2024-12-09,2024-12-06 17:02:00,0001213900-24-106542,0001840317,VEEA INC.,0099
8,http://www.sec.gov/Archives/edgar/data/1193311/000119312524271510/d880088ds4.htm,DQC.US.0099.9533,US GAAP 2024,S-4,2024-12-05,2024-12-05 16:07:00,0001193125-24-271510,0001193311,Oncor Electric Delivery Company LLC,0099
9,http://www.sec.gov/Archives/edgar/data/1409171/000140917124000106/titn-20241031.htm,DQC.US.0099.9533,US GAAP 2024,10-Q,2024-12-05,2024-12-05 16:03:00,0001409171-24-000106,0001409171,TITAN MACHINERY INC.,0099


In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df.to_csv(r"D:\assertions-public-exposure.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('assertions-public-exposure.csv')
#!cp data.csv "drive/My Drive/"

In [77]:
for accession in df['report.accession'].unique():
  url_report = f"https://api.xbrl.us/api/v1/report/search?report.accession={accession}&fields=report.accession,report.creation-software"
  response_report = requests.get(url_report, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
  accession_output = []
  if response_report.status_code == 200:
    report_data = response_report.json()
    #accession_output += report_data['data'][0]
    print(report_data['data'][0])

  elif response_report.status_code == 401:  # Unauthorized, token might have expired
      print("Authorization token expired. Try refreshing it.")
      tokenInfo = refresh(tokenInfo)  # Call your refresh function
  else:
    print(f"Error fetching data for {accession}: Status code {response_report.status_code}, {response_report.text}")

#print(accession_output)

{'report.accession': '0001213900-24-108485', 'report.creation-software': 'Generated by CompSci Transform (tm) - http://www.compsciresources.com'}
{'report.accession': '0001980295-24-000021', 'report.creation-software': 'Novaworks'}
{'report.accession': '0001213900-24-108286', 'report.creation-software': 'Generated by CompSci Transform (tm) - http://www.compsciresources.com'}
{'report.accession': '0001199835-24-000544', 'report.creation-software': 'Novaworks'}
{'report.accession': '0001477932-24-007979', 'report.creation-software': 'XBRL Document Created with XBRLMaster'}
{'report.accession': '0000074046-24-000094', 'report.creation-software': 'XBRL Document Created with the Workiva Platform'}
{'report.accession': '0001634117-24-000130', 'report.creation-software': 'XBRL Document Created with the Workiva Platform'}
{'report.accession': '0001213900-24-106542', 'report.creation-software': 'Generated by CompSci Transform (tm) - http://www.compsciresources.com'}
{'report.accession': '000119

KeyboardInterrupt: 